In [3]:
import csv
import json
import logging
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

# =========================
# Config
# =========================

CSV_PATH = Path(r"C:\Users\Utilisateur\Downloads\esilv (1).csv")  
OUTPUT_DIR = Path("rag/corpus")
HTML_DIR = OUTPUT_DIR / "html"
PDF_DIR = OUTPUT_DIR / "pdf"

HEADERS = {
    "User-Agent": "ESILV-RAG-Bot/1.0 (projet académique)"
}

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("esilv_scraper")


# =========================
# Lecture du CSV
# =========================

def read_urls_from_csv(csv_path: Path):
    """Lit le CSV et retourne une liste unique d'URLs (colonne 0)."""
    urls = []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader, None)  # on ignore l'en-tête
        for row in reader:
            if not row:
                continue
            url = row[0].strip().strip('"')
            if url.startswith("http"):
                urls.append(url)

    # Supprimer les doublons en conservant l'ordre
    unique_urls = list(dict.fromkeys(urls))
    logger.info(f"{len(unique_urls)} URLs trouvées dans le CSV.")
    return unique_urls


# =========================
# Fonctions de scraping
# =========================

def save_html_page(url: str, soup: BeautifulSoup):
    """
    Sauvegarde le texte principal de la page dans corpus/html
    avec l'URL en première ligne.
    """
    HTML_DIR.mkdir(parents=True, exist_ok=True)

    main = soup.find("article") or soup.find("main") or soup.body
    text = main.get_text(separator="\n", strip=True) if main else soup.get_text(separator="\n", strip=True)

    safe_name = url.replace("https://", "").replace("http://", "").replace("/", "_").replace("?", "_")
    file_path = HTML_DIR / f"{safe_name}.txt"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"URL: {url}\n\n")
        f.write(text)

    logger.info(f"HTML sauvegardé: {file_path}")


def extract_pdf_links(url: str, soup: BeautifulSoup):
    """
    Récupère tous les liens PDF présents sur la page.
    """
    pdf_links = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        full = urljoin(url, href)
        if full.lower().endswith(".pdf"):
            pdf_links.add(full)
    return pdf_links


def download_pdf(pdf_url: str):
    """
    Télécharge un PDF dans corpus/pdf + sauve un petit .meta.json
    avec l'URL d'origine.
    """
    PDF_DIR.mkdir(parents=True, exist_ok=True)

    parsed = urlparse(pdf_url)
    filename = parsed.path.split("/")[-1] or "document.pdf"
    file_path = PDF_DIR / filename

    if file_path.exists():
        logger.info(f"PDF déjà présent, skip: {file_path.name}")
    else:
        logger.info(f"Téléchargement PDF: {pdf_url}")
        resp = requests.get(pdf_url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        with open(file_path, "wb") as f:
            f.write(resp.content)

    # On crée un fichier de métadonnées avec l'URL d'origine
    meta_path = file_path.with_suffix(file_path.suffix + ".meta.json")
    meta = {"source_url": pdf_url}
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    return file_path


def scrape_url(url: str):
    """
    Récupère HTML + liens PDF pour une URL donnée.
    """
    try:
        logger.info(f"Scraping {url}")
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()

        soup = BeautifulSoup(resp.text, "html.parser")

        # 1) Sauvegarder le contenu HTML
        save_html_page(url, soup)

        # 2) Télécharger les PDFs trouvés
        pdf_links = extract_pdf_links(url, soup)
        for pdf_url in pdf_links:
            try:
                download_pdf(pdf_url)
            except Exception as e:
                logger.warning(f"Erreur lors du téléchargement PDF {pdf_url}: {e}")

        time.sleep(0.5)  # pour éviter de spammer le site

    except Exception as e:
        logger.warning(f"Erreur sur {url}: {e}")


def main():
    urls = read_urls_from_csv(CSV_PATH)
    for url in urls:
        scrape_url(url)


if __name__ == "__main__":
    main()


INFO:esilv_scraper:25 URLs trouvées dans le CSV.
INFO:esilv_scraper:Scraping https://www.esilv.fr/ufaq/
INFO:esilv_scraper:HTML sauvegardé: rag\corpus\html\www.esilv.fr_ufaq_.txt
INFO:esilv_scraper:Scraping https://www.esilv.fr/ufaq/what-about-housing/
INFO:esilv_scraper:HTML sauvegardé: rag\corpus\html\www.esilv.fr_ufaq_what-about-housing_.txt
INFO:esilv_scraper:Scraping https://www.esilv.fr/ufaq/what-the-average-rent-for-a-flat-in-paris-and-its-suburbs/
INFO:esilv_scraper:HTML sauvegardé: rag\corpus\html\www.esilv.fr_ufaq_what-the-average-rent-for-a-flat-in-paris-and-its-suburbs_.txt
INFO:esilv_scraper:Scraping https://www.esilv.fr/ufaq/how-can-i-apply/
INFO:esilv_scraper:HTML sauvegardé: rag\corpus\html\www.esilv.fr_ufaq_how-can-i-apply_.txt
INFO:esilv_scraper:Scraping https://www.esilv.fr/ufaq/how-can-i-meet-an-admissions-officer-or-a-programme-director/
INFO:esilv_scraper:HTML sauvegardé: rag\corpus\html\www.esilv.fr_ufaq_how-can-i-meet-an-admissions-officer-or-a-programme-directo